In [1]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# ---------- 1. Training data (page-level text, label = document type) ----------
data = [
    # Invoice
    ("Invoice number 4521 date 12 March total amount due 5400 rupees tax GST", "invoice"),
    ("Bill to customer name quantity unit price subtotal payment due in 30 days", "invoice"),
    ("Tax invoice item description amount payable bank account details", "invoice"),
    ("Invoice total balance due please remit payment purchase order number", "invoice"),
    ("Billing statement amount due date invoice id GST number total", "invoice"),
    ("Payment terms net 30 invoice amount tax subtotal shipping charges", "invoice"),
    # Resume
    ("Curriculum vitae objective education B.Tech skills Python experience internship", "resume"),
    ("Work experience software engineer projects skills certifications references", "resume"),
    ("Resume career summary academic qualifications technical skills achievements", "resume"),
    ("Professional experience education degree university skills languages hobbies", "resume"),
    ("Candidate profile objective seeking position skills projects declaration", "resume"),
    ("Experience team lead developer education masters skills Java Python SQL", "resume"),
    # Medical report
    ("Patient name diagnosis blood pressure prescription doctor hospital treatment", "medical"),
    ("Lab report hemoglobin glucose cholesterol test results patient age", "medical"),
    ("Discharge summary patient admitted symptoms medication dosage follow up", "medical"),
    ("Clinical report diagnosis physician prescribed tablets twice daily patient", "medical"),
    ("Radiology report scan findings patient history doctor impression", "medical"),
    ("Medical certificate patient treated hospital rest advised medication", "medical"),
    # Legal / contract
    ("This agreement is made between the parties hereinafter referred to as terms", "legal"),
    ("Contract clause liability indemnity jurisdiction governing law witness", "legal"),
    ("Non disclosure agreement confidential information party obligations term", "legal"),
    ("Lease agreement tenant landlord rent security deposit termination clause", "legal"),
    ("Affidavit sworn before notary court petitioner respondent hereby declare", "legal"),
    ("Legal notice advocate client breach of contract demand within days", "legal"),
]
texts, labels = zip(*data)

# ---------- 2. Preprocessing ----------
def clean(t):
    t = t.lower()
    t = re.sub(r"[^a-z\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

texts = [clean(t) for t in texts]

# ---------- 3. Train / test ----------
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels)

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))

# ---------- 4. Document separation ----------
# A scanned batch: pages of different documents mixed in one stream
batch_pages = [
    "Invoice number 9001 total amount due GST payment terms",
    "Item description quantity unit price subtotal tax amount payable",
    "Curriculum vitae objective education skills Python experience",
    "Work experience software developer projects certifications skills",
    "Professional experience internship education degree references",
    "Patient diagnosis prescription doctor hospital treatment",
    "Lab report glucose cholesterol test results patient",
    "This agreement is made between the parties terms clause liability",
    "Governing law jurisdiction indemnity witness signed parties",
]

page_labels = model.predict([clean(p) for p in batch_pages])
page_probs = model.predict_proba([clean(p) for p in batch_pages]).max(axis=1)

# Start a new document whenever the predicted class changes
documents = []
current = {"type": page_labels[0], "pages": [0]}
for i in range(1, len(batch_pages)):
    if page_labels[i] == current["type"]:
        current["pages"].append(i)
    else:
        documents.append(current)
        current = {"type": page_labels[i], "pages": [i]}
documents.append(current)

print("\nPage-level predictions:")
for i, (lab, pr) in enumerate(zip(page_labels, page_probs)):
    print(f"  Page {i+1}: {lab:<10} (confidence {pr:.2f})")

print("\nSeparated documents:")
for n, d in enumerate(documents, 1):
    print(f"  Document {n}: type = {d['type']:<8} pages = {[p+1 for p in d['pages']]}")

Accuracy: 1.0
              precision    recall  f1-score   support

     invoice       1.00      1.00      1.00         2
       legal       1.00      1.00      1.00         1
     medical       1.00      1.00      1.00         2
      resume       1.00      1.00      1.00         1

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6


Page-level predictions:
  Page 1: invoice    (confidence 0.39)
  Page 2: invoice    (confidence 0.37)
  Page 3: resume     (confidence 0.48)
  Page 4: resume     (confidence 0.41)
  Page 5: resume     (confidence 0.42)
  Page 6: medical    (confidence 0.39)
  Page 7: medical    (confidence 0.39)
  Page 8: legal      (confidence 0.43)
  Page 9: legal      (confidence 0.40)

Separated documents:
  Document 1: type = invoice  pages = [1, 2]
  Document 2: type = resume   pages = [3, 4, 5]
  Document 3: type = medical  pages = [6, 7]
  Document 4: t